# 01. Safe Serialization Weights Only Safetensors and Mmap

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** Hugging Face (Safetensors, 2022), PyTorch Security Team (Weights Only Mode, 2023).
- **Purpose:** Eliminate arbitrary remote code execution risks in Python pickle and achieve instant, zero-copy model loading via safetensors and memory mapping.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Distributed and Scalable Master Track](../../README.md#16_distributed_and_scalable) $\to$ `🔗 Downstream Syntheses:` [ONNX Export Dynamo and ONNXRuntime Inference](02_onnx_export_dynamo_and_onnxruntime_inference.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never use legacy Python pickle (`torch.save` / `torch.load` with `weights_only=False`) when loading checkpoints from public repositories or external contributors. Deserializing untrusted pickle files exposes host clusters to instantaneous arbitrary remote code execution (RCE).

## Mathematical Foundations & Governing Equations

$$
\text{WireLayout} = \text{UInt64}(\text{header\_len}) \parallel \text{JSON}(\text{metadata}) \parallel \text{DataBuffers}, \quad T_{\text{cold\_start}} = O(1)
$$

### Asymptotic Time & Memory Complexity
- **Zero RAM Doubling:** Memory-mapping maps disk blocks directly into virtual address space, eliminating the $2\times$ CPU memory duplication of eager loading.
- **Linear Parsing Complexity:** Validating the pure JSON header takes $O(K)$ time for $K$ tensors with zero arbitrary Python bytecode execution.

In [ ]:
import json
import struct
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def safetensors_pack_tensors(tensors: dict[str, torch.Tensor]) -> tuple[bytes, int]:
    header = {}
    offset = 0
    payload_buffers = []
    for name, tensor in tensors.items():
        t_bytes = tensor.detach().contiguous().cpu().numpy().tobytes()
        header[name] = {"dtype": "F32", "shape": list(tensor.shape), "data_offsets": [offset, offset + len(t_bytes)]}
        offset += len(t_bytes)
        payload_buffers.append(t_bytes)
    header_json = json.dumps(header, separators=(",", ":")).encode("utf-8")
    wire_bytes = struct.pack("<Q", len(header_json)) + header_json + b"".join(payload_buffers)
    return wire_bytes, len(header_json)

def safetensors_unpack_tensors(wire_bytes: bytes) -> dict[str, torch.Tensor]:
    header_len = struct.unpack("<Q", wire_bytes[:8])[0]
    header = json.loads(wire_bytes[8:8+header_len].decode("utf-8"))
    data_start = 8 + header_len
    unpacked = {}
    for name, meta in header.items():
        chunk = wire_bytes[data_start + meta["data_offsets"][0] : data_start + meta["data_offsets"][1]]
        unpacked[name] = torch.frombuffer(bytearray(chunk), dtype=torch.float32).reshape(meta["shape"]).clone()
    return unpacked

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

y_ref = weights["x_query"] @ weights["w_proj"] + weights["b_proj"]
assert float(torch.max(torch.abs(y_ref - weights["y_ref"]))) < 1e-5

wire_bytes, header_len = safetensors_pack_tensors({"w_proj": weights["w_proj"], "b_proj": weights["b_proj"]})
unpacked = safetensors_unpack_tensors(wire_bytes)
y_unpacked = weights["x_query"] @ unpacked["w_proj"] + unpacked["b_proj"]
assert float(torch.max(torch.abs(y_unpacked - weights["y_ref"]))) < 1e-5
print("All Safe Serialization and Safetensors Wire Format parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

test_dict = {"w_proj": weights["w_proj"], "b_proj": weights["b_proj"]}
wire_bytes, _ = safetensors_pack_tensors(test_dict)

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: safetensors_pack_tensors(test_dict), iters=50, name="Safetensors Pack"),
    benchmark_fn(lambda: safetensors_unpack_tensors(wire_bytes), iters=50, name="Safetensors Unpack"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Legacy Pickle Host RAM Doubling vs Zero-Copy Safetensors Mmap
model_params = 70_000_000_000  # 70B parameters
bytes_per_param = 2            # FP16
model_size_gb = (model_params * bytes_per_param) / (1024 ** 3)  # 130.39 GB

# Legacy torch.load (pickle) allocates a duplicate full copy in CPU host RAM before GPU transfer:
pickle_peak_host_ram_gb = 2 * model_size_gb

# Safetensors with mmap=True maps pages directly into virtual memory (0 extra host RAM):
safetensors_mmap_ram_gb = 0.05  # only metadata header

print(f"Model Size (FP16):                    {model_size_gb:.2f} GB")
print(f"Legacy Pickle Peak Host RAM:          {pickle_peak_host_ram_gb:.2f} GB (Severe Host OOM Risk!)")
print(f"Safetensors Mmap Overhead:            {safetensors_mmap_ram_gb:.2f} GB (Near-Zero Memory Footprint)")
assert pickle_peak_host_ram_gb > 250.0, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Exploit Surface Elimination:** The declarative JSON header format of `safetensors` completely neutralizes arbitrary Python bytecode injection attacks.
2. **Zero-Copy Operating System Page Faulting:** Memory-mapping files directly into virtual memory eliminates duplicate host memory allocations.
3. **Transitive Interoperability:** A single `.safetensors` file runs natively across PyTorch 2.x, JAX, Rust Candle, and C++ runtimes without conversion.

---
### Cross-Domain Generalisation Matrix
| Domain | Declarative Serialization Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Financial High-Frequency Trading** | Simple Binary Encoding (SBE) / FlatBuffers | Eliminates serialization parsing overhead via zero-copy struct casting |
| **Geospatial & Climate Mapping** | Cloud-Optimized GeoTIFF (COG) with byte-range requests | Streams specific raster tiles on-demand without downloading multi-gigabyte files |
| **Embedded Firmware Updates** | Cryptographically signed declarative binary manifests | Guarantees firmware payload authenticity before flashing microcontroller ROM |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_safe_serialization_weights_only_safetensors_and_mmap.md](../../markdown/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.md)
- **PyTorch 2.x Module:** [01_safe_serialization_weights_only_safetensors_and_mmap.py](../../pytorch2/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.py)
- **Functional JAX Module:** [01_safe_serialization_weights_only_safetensors_and_mmap.py](../../jax/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.py)
- **Native Rust Candle Track:** [01_safe_serialization_weights_only_safetensors_and_mmap.rs](../../candle/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.rs)
- **Hardware Kernel (CPU / WGSL):** [01_safe_serialization_weights_only_safetensors_and_mmap.rs](../../kernels/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.rs)
- **Golden SafeTensors:** [01_safe_serialization_weights_only_safetensors_and_mmap.safetensors](../../weights/17_deployment_low_level_and_capstones/01_safe_serialization_weights_only_safetensors_and_mmap.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 16. Distributed and Scalable](../../README.md#16_distributed_and_scalable) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 02. ONNX Export Dynamo and ONNXRuntime Inference](02_onnx_export_dynamo_and_onnxruntime_inference.ipynb) &rarr;